In [2]:
import numpy as np
import random
import copy
import time


# ==========================
# Instância do Sudoku
# ==========================

sudoku = np.array([
    [6, 0, 0, 0, 8, 0, 0, 0, 0],
    [0, 0, 9, 7, 0, 0, 0, 2, 8],
    [2, 0, 0, 0, 0, 0, 3, 0, 0],

    [0, 0, 0, 0, 0, 0, 0, 8, 0],
    [0, 0, 0, 4, 5, 0, 0, 9, 0],
    [0, 9, 3, 0, 2, 0, 0, 0, 1],

    [0, 0, 0, 6, 0, 0, 2, 0, 0],
    [4, 0, 6, 0, 0, 9, 0, 0, 0],
    [0, 0, 2, 3, 0, 0, 0, 0, 4]
])

# =====================================
# Controle de avaliações
# =====================================

MAX_AVALIACOES = 400000
AVALIACOES = 0

# ==========================
# Informações da instância
# ==========================

# Posições originalmente preenchidas
posicoes_fixas = list(zip(*np.where(sudoku != 0)))

# Posições vazias
posicoes_vazias = list(zip(*np.where(sudoku == 0)))

# Número de variáveis de decisão
N = len(posicoes_vazias)

print(f"Células fixas : {len(posicoes_fixas)}")
print(f"Células vazias: {N}")

Células fixas : 24
Células vazias: 57


In [3]:
# =====================================
# Função de Aptidão
# =====================================

def contar_conflitos_pares(valores):
    """
    Conta todos os pares de valores iguais em um vetor.

    Exemplo:
    [5, 5, 5] possui 3 pares conflitantes:
    (5_1, 5_2), (5_1, 5_3) e (5_2, 5_3).
    """

    valores = valores[valores != 0]
    conflitos = 0

    for j in range(len(valores) - 1):
        for l in range(j + 1, len(valores)):
            if valores[j] == valores[l]:
                conflitos += 1

    return conflitos


def calcular_fitness(tabuleiro):
    """
    Calcula a quantidade total de conflitos
    nas linhas, colunas e blocos 3x3.

    Cada par de valores iguais é contabilizado
    como um conflito.

    Quanto menor o fitness, melhor a solução.
    O valor ótimo é fitness igual a zero.

    Cada chamada da função corresponde
    a uma avaliação da função objetivo.
    """
    global AVALIACOES

    # Conta uma avaliação da função objetivo
    AVALIACOES += 1

    tabuleiro = np.asarray(tabuleiro)

    if tabuleiro.shape != (9, 9):
        raise ValueError("O tabuleiro deve possuir dimensão 9x9.")

    conflitos = 0

    # ---------- Linhas ----------
    for linha in tabuleiro:
        conflitos += contar_conflitos_pares(linha)

    # ---------- Colunas ----------
    for coluna in tabuleiro.T:
        conflitos += contar_conflitos_pares(coluna)

    # ---------- Blocos 3x3 ----------
    for i in range(0, 9, 3):
        for j in range(0, 9, 3):
            bloco = tabuleiro[i:i + 3, j:j + 3].flatten()
            conflitos += contar_conflitos_pares(bloco)

    return conflitos


# =====================================
# Funções auxiliares
# =====================================

def copiar_tabuleiro(tabuleiro):
    """
    Retorna uma cópia independente do tabuleiro.
    """

    return copy.deepcopy(tabuleiro)


def vetor_para_tabuleiro(vetor, sudoku_original):
    """
    Converte um vetor de decisão em um tabuleiro de Sudoku.

    Cada valor do vetor é inserido em uma posição
    originalmente vazia do Sudoku.
    """

    if len(vetor) != len(posicoes_vazias):
        raise ValueError(
            "O tamanho do vetor deve ser igual ao número de posições vazias."
        )

    tabuleiro = sudoku_original.copy()

    for valor, (i, j) in zip(vetor, posicoes_vazias):
        tabuleiro[i, j] = valor

    return tabuleiro


def mostrar_tabuleiro(tabuleiro):
    """
    Exibe o tabuleiro de forma organizada.
    """

    for i in range(9):
        if i > 0 and i % 3 == 0:
            print("-" * 21)

        linha_formatada = []

        for j in range(9):
            if j > 0 and j % 3 == 0:
                linha_formatada.append("|")

            linha_formatada.append(str(tabuleiro[i, j]))

        print(" ".join(linha_formatada))


# =====================================
# Teste da função de aptidão
# =====================================

print("Fitness inicial:", calcular_fitness(sudoku))
print("\nTabuleiro inicial:\n")

mostrar_tabuleiro(sudoku)

Fitness inicial: 0

Tabuleiro inicial:

6 0 0 | 0 8 0 | 0 0 0
0 0 9 | 7 0 0 | 0 2 8
2 0 0 | 0 0 0 | 3 0 0
---------------------
0 0 0 | 0 0 0 | 0 8 0
0 0 0 | 4 5 0 | 0 9 0
0 9 3 | 0 2 0 | 0 0 1
---------------------
0 0 0 | 6 0 0 | 2 0 0
4 0 6 | 0 0 9 | 0 0 0
0 0 2 | 3 0 0 | 0 0 4


In [4]:
# ============================================================
# CÉLULA 4 — INTEGER PARTICLE SWARM OPTIMIZATION (IPSO)
# ============================================================


def funcao_escala_ipso(velocidade):
    """
    Converte a velocidade em probabilidade de alteração
    da posição, conforme a função de escala simétrica:

        S(v) = 2 / (1 + exp(-|v|)) - 1

    Propriedades:
    - S(0) = 0;
    - S(v) tende a 1 quando |v| aumenta;
    - o retorno pertence ao intervalo [0, 1).
    """

    return 2.0 / (1.0 + np.exp(-np.abs(velocidade))) - 1.0


def avaliar_particula_ipso(particula, sudoku_original):
    """
    Converte o vetor da partícula em um tabuleiro completo
    e calcula sua função de aptidão.

    A partícula contém apenas os valores das células
    originalmente vazias. As células fixas permanecem
    inalteradas.
    """

    tabuleiro = vetor_para_tabuleiro(
        particula,
        sudoku_original
    )

    fitness = calcular_fitness(tabuleiro)

    return fitness


def executar_ipso(
    sudoku_original,
    quantidade_particulas=60,
    c1=0.25,
    c2=0.25,
    fator_constricao=0.7298,
    velocidade_min=-2.0,
    velocidade_max=2.0,
    limite_estagnacao=1000,
    exibir_progresso=True,
    intervalo_exibicao=1000
):
    """
    Executa o algoritmo IPSO para resolução do Sudoku.

    Representação:
    --------------
    Cada partícula é um vetor de inteiros de dimensão N,
    sendo N o número de células originalmente vazias.

    Cada posição do vetor contém um valor inteiro entre 1 e 9.

    Atualização da velocidade:
    --------------------------
    v(n+1) = K * [
        v(n)
        + c1*r1*(pbest - p)
        + c2*r2*(gbest - p)
    ]

    Atualização da posição:
    -----------------------
    Para cada dimensão:

    - calcula-se S(v);
    - sorteia-se um número aleatório em [0,1);
    - se o número for menor que S(v), a posição recebe
      um novo inteiro entre 1 e 9;
    - caso contrário, o valor atual é mantido.

    Critério de parada:
    -------------------
    - fitness igual a zero; ou
    - número máximo de avaliações da função de aptidão.
    """

    # --------------------------------------------------------
    # Validação dos parâmetros
    # --------------------------------------------------------

    if quantidade_particulas <= 0:
        raise ValueError(
            "A quantidade de partículas deve ser positiva."
        )

    if velocidade_min >= velocidade_max:
        raise ValueError(
            "velocidade_min deve ser menor que velocidade_max."
        )

    if limite_estagnacao <= 0:
        raise ValueError(
            "O limite de estagnação deve ser positivo."
        )

    # Marca o início da execução
    tempo_inicial = time.perf_counter()

    # Número de dimensões do problema:
    # uma dimensão para cada célula vazia
    dimensao = len(posicoes_vazias)

    # --------------------------------------------------------
    # Inicialização das posições
    # --------------------------------------------------------

    # Cada partícula recebe valores inteiros aleatórios
    # pertencentes ao intervalo [1, 9]
    posicoes = np.random.randint(
        low=1,
        high=10,
        size=(quantidade_particulas, dimensao)
    )

    # --------------------------------------------------------
    # Inicialização das velocidades
    # --------------------------------------------------------

    # As velocidades são inicialmente iguais a zero
    velocidades = np.zeros(
        shape=(quantidade_particulas, dimensao),
        dtype=float
    )

    # --------------------------------------------------------
    # Avaliação da população inicial
    # --------------------------------------------------------

    fitness_atual = np.array([
        avaliar_particula_ipso(
            particula,
            sudoku_original
        )
        for particula in posicoes
    ])

    # --------------------------------------------------------
    # Inicialização do pbest
    # --------------------------------------------------------

    # No início, a melhor posição conhecida por cada partícula
    # é sua própria posição inicial
    pbest = posicoes.copy()

    fitness_pbest = fitness_atual.copy()

    # --------------------------------------------------------
    # Registro do melhor resultado global
    # --------------------------------------------------------

    melhor_indice = int(np.argmin(fitness_pbest))

    melhor_posicao_global = pbest[melhor_indice].copy()

    melhor_fitness_global = int(
        fitness_pbest[melhor_indice]
    )

    # Histórico do menor fitness encontrado
    historico_fitness = [melhor_fitness_global]

    # Conta quantas iterações ocorreram sem melhoria
    iteracoes_sem_melhora = 0

    # Guarda a última iteração realmente executada
    iteracao_final = 0

    # Contador de iterações
    iteracao = 0

    # --------------------------------------------------------
    # Laço principal do IPSO
    # --------------------------------------------------------

    # Só inicia uma nova iteração se ainda houver orçamento
    # suficiente para avaliar todas as partículas.
    while AVALIACOES + quantidade_particulas <= MAX_AVALIACOES:

        iteracao += 1
        iteracao_final = iteracao

        # ----------------------------------------------------
        # Melhor posição global
        # ----------------------------------------------------

        # A melhor posição global é compartilhada
        # com todas as partículas.
        gbest = np.tile(
            melhor_posicao_global,
            (quantidade_particulas, 1)
        )

        # ----------------------------------------------------
        # Geração dos coeficientes aleatórios r1 e r2
        # ----------------------------------------------------

        # São gerados valores independentes para cada partícula
        # e para cada dimensão.
        r1 = np.random.uniform(
            low=0.0,
            high=1.0,
            size=(quantidade_particulas, dimensao)
        )

        r2 = np.random.uniform(
            low=0.0,
            high=1.0,
            size=(quantidade_particulas, dimensao)
        )

        # ----------------------------------------------------
        # Atualização da velocidade
        # ----------------------------------------------------

        componente_cognitivo = (
            c1
            * r1
            * (pbest - posicoes)
        )

        componente_social = (
            c2
            * r2
            * (gbest - posicoes)
        )

        velocidades = fator_constricao * (
            velocidades
            + componente_cognitivo
            + componente_social
        )

        # Limita as velocidades ao intervalo estabelecido
        velocidades = np.clip(
            velocidades,
            velocidade_min,
            velocidade_max
        )

        # ----------------------------------------------------
        # Conversão da velocidade em probabilidade
        # ----------------------------------------------------

        probabilidades = funcao_escala_ipso(
            velocidades
        )

        # ----------------------------------------------------
        # Atualização das posições
        # ----------------------------------------------------

        # Para cada dimensão, gera-se um número aleatório
        # para decidir se aquela posição será alterada.
        numeros_aleatorios = np.random.uniform(
            low=0.0,
            high=1.0,
            size=(quantidade_particulas, dimensao)
        )

        # True indica que a posição será substituída
        mascara_alteracao = (
            numeros_aleatorios < probabilidades
        )

        # Sorteia possíveis novos valores entre 1 e 9
        novos_valores = np.random.randint(
            low=1,
            high=10,
            size=(quantidade_particulas, dimensao)
        )

        # Quando a máscara é verdadeira, utiliza o novo valor.
        # Caso contrário, mantém o valor atual.
        posicoes = np.where(
            mascara_alteracao,
            novos_valores,
            posicoes
        )

        # ----------------------------------------------------
        # Avaliação das novas posições
        # ----------------------------------------------------

        fitness_atual = np.array([
            avaliar_particula_ipso(
                particula,
                sudoku_original
            )
            for particula in posicoes
        ])

        # ----------------------------------------------------
        # Atualização do pbest
        # ----------------------------------------------------

        # Uma partícula atualiza seu pbest apenas quando
        # encontra uma posição com menor fitness.
        mascara_melhora_pbest = (
            fitness_atual < fitness_pbest
        )

        pbest[mascara_melhora_pbest] = (
            posicoes[mascara_melhora_pbest].copy()
        )

        fitness_pbest[mascara_melhora_pbest] = (
            fitness_atual[mascara_melhora_pbest]
        )

        # ----------------------------------------------------
        # Atualização do melhor resultado global
        # ----------------------------------------------------

        melhor_indice_iteracao = int(
            np.argmin(fitness_pbest)
        )

        melhor_fitness_iteracao = int(
            fitness_pbest[melhor_indice_iteracao]
        )

        if melhor_fitness_iteracao < melhor_fitness_global:

            melhor_fitness_global = (
                melhor_fitness_iteracao
            )

            melhor_posicao_global = pbest[
                melhor_indice_iteracao
            ].copy()

            # Houve melhoria: reinicia o contador
            iteracoes_sem_melhora = 0

        else:

            iteracoes_sem_melhora += 1

        # ----------------------------------------------------
        # Reinicialização após estagnação
        # ----------------------------------------------------

        # Caso não haja melhoria pelo número definido de
        # iterações consecutivas, as velocidades são zeradas.
        if iteracoes_sem_melhora >= limite_estagnacao:

            velocidades.fill(0.0)

            iteracoes_sem_melhora = 0

        # Registra o melhor fitness encontrado até o momento
        historico_fitness.append(
            melhor_fitness_global
        )

        # ----------------------------------------------------
        # Exibição do progresso
        # ----------------------------------------------------

        if exibir_progresso:

            deve_exibir = (
                iteracao == 1
                or iteracao % intervalo_exibicao == 0
                or melhor_fitness_global == 0
            )

            if deve_exibir:

                print(
                    f"Iteração {iteracao:>7} | "
                    f"Melhor fitness: {melhor_fitness_global} | "
                    f"Avaliações: {AVALIACOES}"
                )

        # ----------------------------------------------------
        # Critério de parada
        # ----------------------------------------------------

        # Se encontrou uma solução válida, encerra
        # mesmo que ainda reste orçamento de avaliações.
        if melhor_fitness_global == 0:
            break

    # --------------------------------------------------------
    # Conversão da melhor partícula para tabuleiro
    # --------------------------------------------------------

    melhor_tabuleiro = vetor_para_tabuleiro(
        melhor_posicao_global,
        sudoku_original
    )

    # Tempo total de execução
    tempo_execucao = (
        time.perf_counter() - tempo_inicial
    )

    # --------------------------------------------------------
    # Retorno padronizado
    # --------------------------------------------------------

    resultado = {
        "algoritmo": "IPSO",
        "tabuleiro": melhor_tabuleiro,
        "fitness": melhor_fitness_global,
        "iteracoes": iteracao_final,
        "avaliacoes": AVALIACOES,
        "tempo": tempo_execucao,
        "historico": historico_fitness,
        "solucionado": melhor_fitness_global == 0
    }

    return resultado

In [ ]:
# ============================================================
# CÉLULA 5 — 30 EXECUÇÕES DO IPSO
# ============================================================

resultados_ipso = []

for execucao in range(30):

    # Uma semente diferente para cada execução
    SEED_EXECUCAO = execucao

    random.seed(SEED_EXECUCAO)
    np.random.seed(SEED_EXECUCAO)

    PARAMETROS_IPSO = {
        "quantidade_particulas": 60,
        "c1": 0.55,
        "c2": 0.55,
        "fator_constricao": 0.650,
        "velocidade_min": -3.0,
        "velocidade_max": 3.0,
        "limite_estagnacao": 500,

        # Melhor deixar False nas 30 execuções
        # para não gerar milhares de linhas no Colab
        "exibir_progresso": False,
        "intervalo_exibicao": 500
    }

    # Reinicia o contador global antes de cada execução
    AVALIACOES = 0

    resultado = executar_ipso(
        sudoku_original=sudoku,
        **PARAMETROS_IPSO
    )

    # Guarda os dados importantes da execução
    resultados_ipso.append({
        "execucao": execucao + 1,
        "seed": SEED_EXECUCAO,
        "fitness": resultado["fitness"],
        "iteracoes": resultado["iteracoes"],
        "avaliacoes": resultado["avaliacoes"],
        "tempo": resultado["tempo"],
        "solucionado": resultado["solucionado"]
    })

    # Mostra apenas um resumo de cada execução
    print(
        f"Execução {execucao + 1:02d} | "
        f"Seed: {SEED_EXECUCAO:02d} | "
        f"Fitness: {resultado['fitness']} | "
        f"Iterações: {resultado['iteracoes']} | "
        f"Avaliações: {resultado['avaliacoes']} | "
        f"Tempo: {resultado['tempo']:.2f}s | "
        f"Solucionado: {resultado['solucionado']}"
    )

In [ ]:
# ============================================================
# RESUMO DAS 30 EXECUÇÕES
# ============================================================

fitness = [r["fitness"] for r in resultados_ipso]
tempos = [r["tempo"] for r in resultados_ipso]
avaliacoes = [r["avaliacoes"] for r in resultados_ipso]

sucessos = sum(r["solucionado"] for r in resultados_ipso)

taxa_sucesso = (sucessos / len(resultados_ipso)) * 100

print("\n" + "=" * 50)
print("RESUMO — IPSO")
print("=" * 50)

print(f"Execuções: {len(resultados_ipso)}")
print(f"Sucessos: {sucessos}")
print(f"Taxa de sucesso: {taxa_sucesso:.2f}%")

print(f"\nMelhor fitness: {min(fitness)}")
print(f"Pior fitness: {max(fitness)}")
print(f"Fitness médio: {np.mean(fitness):.2f}")
print(f"Mediana do fitness: {np.median(fitness):.2f}")
print(f"Desvio padrão do fitness: {np.std(fitness):.2f}")

print(f"\nTempo médio: {np.mean(tempos):.2f} segundos")
print(f"Mediana do tempo: {np.median(tempos):.2f} segundos")

print(f"\nMédia de avaliações: {np.mean(avaliacoes):.2f}")


RESUMO — IPSO
Execuções: 30
Sucessos: 0
Taxa de sucesso: 0.00%

Melhor fitness: 10
Pior fitness: 22
Fitness médio: 17.27
Mediana do fitness: 17.00
Desvio padrão do fitness: 2.95

Tempo médio: 107.62 segundos
Mediana do tempo: 108.25 segundos

Média de avaliações: 399960.00
